In [2]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path().resolve().parent  # notebooks/ -> parent is project root
sys.path.append(str(PROJECT_ROOT))

In [3]:
import pandas as pd
import numpy as np
from pandas import DataFrame
import os


from src.data.utils import lazy

In [129]:
class FeatureBuilder:
    """
    `FeatureBuilder` for NCAA M/W Basketball data
    """
    #constants
    stat_cols = [
        'Score',
        'FGM',
        'FGA',
        'FGM2',
        'FGA2',
        'FGM3',
        'FGA3',
        'FTM',
        'FTA',
        'OR',
        'DR',
        'Ast',
        'TO',
        'Stl',
        'Blk',
        'PF',
        'Pace'
#         'Poss'
    ]
    
    other_cols = [
        'Season',
        'DayNum',
        'Loc',
        'NumOT',
        'team_TeamID',
        'opponent_TeamID',
        'Tournament',
        'GameID',
        'Minutes'
    ]
    
    def __init__(self, year: int=2025, men: bool=True, rolling_window: int=30, feature_methods: list=None) -> None:
        """
        Initializes `FeatureBuilder` for a certain data pull year and gender

        Args:
            year: Year of the data pull
            men: boolean on if you want Mens (`True`) or Women's (`False`)

        Returns:
            `None`, but initilaizes dataloader object
        """
        self.year = year
        self.gender = "M" if men else "W"
        self.PROJECT_ROOT = Path().resolve().parent  # /src/data -> project root
        self.rolling_window = rolling_window
        self.feature_cols = self._get_feature_cols()
        
        if feature_methods is None:
            feature_methods = ["avg"]
        self.feature_methods = feature_methods
        
    @lazy
    def data(self) -> DataFrame:
        path = (
                self.PROJECT_ROOT
                / "data"
                / "interim"
                / f"{self.gender}{self.year}_formatted_data.csv"
            )
        if not os.path.exists(path):
            raise ValueError(f"{path} does not exist. Please run the DataLoader and write the processed data first")
        return pd.read_csv(path).sort_values(["Season", "DayNum", "GameID"]).reset_index(drop=True) 
    
    @lazy
    def per_possession(self) -> DataFrame:
        df = self.data.copy()
        df = self._build_advanced_features(df)
        for col in self.stat_cols:
            if col=="Pace":
                continue
            for team in ['team', 'opponent']:
                df[f'{team}_{col}'] = df[f'{team}_{col}']/df.Possessions
        df[f'team_Pace'] = df.Possessions/df.Minutes
        df[f'opponent_Pace'] = df.Possessions/df.Minutes
        return df
    
    def _build_advanced_features(self, df):
        box_score = df.copy()
        
        box_score['team_FGM2'] = box_score.team_FGM - box_score.team_FGM3
        box_score['opponent_FGM2'] = box_score.opponent_FGM - box_score.opponent_FGM3
        box_score['team_FGA2'] = box_score.team_FGA - box_score.team_FGA3
        box_score['opponent_FGA2'] = box_score.opponent_FGA - box_score.opponent_FGA3
        
        box_score['team_FGA2_rate'] = box_score.team_FGA2 / box_score.team_FGA
        box_score['opponent_FGA2_rate'] = box_score.opponent_FGA2 / box_score.opponent_FGA
        # FGA3/FGA
        box_score['team_FGA3_rate'] = box_score.team_FGA3 / box_score.team_FGA
        box_score['opponent_FGA3_rate'] = box_score.opponent_FGA3 / box_score.opponent_FGA
        
        box_score['team_FGMiss'] = box_score.team_FGA - box_score.team_FGM 
        box_score['opponent_FGMiss'] = box_score.opponent_FGA - box_score.opponent_FGM 

        box_score['team_OR_rate'] = box_score.team_OR / box_score.team_FGMiss
        box_score['opponent_OR_rate'] = box_score.opponent_OR / box_score.opponent_FGMiss

        box_score['team_DR_rate'] = box_score.team_DR / box_score.opponent_FGMiss
        box_score['opponent_DR_rate'] = box_score.opponent_OR / box_score.team_FGMiss

        # BLK/oFGA
        box_score['team_Blk_rate'] = box_score.team_Blk / box_score.opponent_FGA
        box_score['opponent_Blk_rate'] = box_score.opponent_Blk / box_score.team_FGA

        # FG2M/FG2A
        box_score['team_FGM2_rate'] = box_score.team_FGM2 / box_score.team_FGA2
        box_score['opponent_FGM2_rate'] = box_score.opponent_FGM2 / box_score.opponent_FGA2

        # FG3M/FG3A
        box_score['team_FGM3_rate'] = box_score.team_FGM3 / box_score.team_FGA3
        box_score['opponent_FGM3_rate'] = box_score.opponent_FGM3 / box_score.opponent_FGA3
        # Assist/FGM
        box_score['team_Ast_rate'] = box_score.team_Ast / box_score.team_FGA
        box_score['opponent_Ast_rate'] = box_score.opponent_Ast / box_score.opponent_FGA


        # FTM/FTA
        box_score['team_FT_rate'] = box_score.team_FTM / box_score.team_FTA
        box_score['opponent_FT_rate'] = box_score.opponent_FTM / box_score.opponent_FTA

        # FTM/FGA
        # effective stats
        # effective FTRate
        box_score['team_eFTM'] = box_score.team_FTM / box_score.team_FGA
        box_score['opponent_eFTM'] = box_score.opponent_FTM / box_score.opponent_FGA

        # effective FG: (FGM+.5*3FGM)/FGA
        box_score['team_eFG_rate'] = (box_score.team_FGM + 0.5*box_score.team_FGM3) / box_score.team_FGA
        box_score['opponent_eFG_rate'] = (box_score.opponent_FGM + 0.5*box_score.opponent_FGM3) / box_score.opponent_FGA

        # PPWS = Points/(FGA+(0.475*FTA))
        box_score['team_PPWS'] = box_score.team_Score/(box_score.team_FGA + (.475 * box_score.team_FTA))
        box_score['opponent_PPWS'] = box_score.opponent_Score/(box_score.opponent_FGA + (.475 * box_score.opponent_FTA))
        
        advanced_stats_cols = ['team_FGA2_rate', 'opponent_FGA2_rate', 'team_FGA3_rate',
            'opponent_FGA3_rate', 'team_OR_rate', 'opponent_OR_rate',
            'team_DR_rate', 'opponent_DR_rate', 'team_Blk_rate',
            'opponent_Blk_rate', 'team_FGM2_rate','opponent_FGM2_rate',
            'team_FGM3_rate', 'opponent_FGM3_rate', 'team_Ast_rate',
            'opponent_Ast_rate', 'team_FT_rate', 'opponent_FT_rate',
            'team_eFTM', 'opponent_eFTM', 'team_eFG_rate',
            'opponent_eFG_rate', 'team_PPWS', 'opponent_PPWS']
        
        box_score[advanced_stats_cols] = box_score[advanced_stats_cols].fillna(0) # rare divide by 0
        self.feature_cols.extend(
            advanced_stats_cols
        )
        
        return box_score.drop(columns=['team_FGMiss', 'opponent_FGMiss'])

    
    def _get_feature_cols(self) -> list:
        """
        Flexible selector for all engineered feature columns.

        include:
            "raw"         - raw stats like FGM, PF, TO
            "per_poss"    - per-possession versions of raw stats
            "all"         - everything
        """

        cols = []

        for stat in self.stat_cols:
            cols.append(f"team_{stat}")
            cols.append(f"opponent_{stat}")

        return cols

    @lazy
    def season_averages(self) -> DataFrame:
        """
        League-wide seasonal averages for per-possession stats.
        Used for shrinkage on low sample sizes.
        """
        df = self.per_possession.copy()

        
        league = (
            df.groupby("Season")[self.feature_cols]
            .mean()
            .reset_index()
        )
        return league
    
    @lazy
    def season_team_averages(self) -> DataFrame:
        """
        League-wide seasonal averages for per-possession stats.
        Used for shrinkage on low sample sizes.
        """
        df = self.per_possession.copy()

        # 1. Make league per-team per-season averages
        league = (
            df.groupby(["Season", "team_TeamID"])[self.feature_cols]
            .mean()
            .reset_index()
        )#.set_index("Season")

        # 2. Make season-wide averages
        season_averages = self.season_averages.copy().set_index("Season")

        # 3. Merge league with season averages on Season
        league = league.merge(
            season_averages[self.feature_cols],
            left_on="Season",
            right_index=True,
            suffixes=("", "_season")
        )

        # 4. Compute simple average (regression to mean)
        for col in self.feature_cols:
            league[col] = (league[col] + league[f"{col}_season"]) / 2

        # 5. Optionally drop the "_season" columns
        league.drop(columns=[f"{c}_season" for c in self.feature_cols], inplace=True)
        return league

    
    
    def make_recency_weight(self, df):
        df = df.copy()
        
        min_day = (df.groupby("Season")["DayNum"]
            .min()
            .reset_index()
            .set_index('Season')
        ) 
        
        
        lin_func = np.linspace(1, 5, 17)/5
        recency_func = lambda x: lin_func.max() if x >= len(lin_func) else lin_func[x]
        df["weight"] = (((df.DayNum - df.Season.map(min_day['DayNum']))//7)
                       ).apply(recency_func)
        return df

    
    def get_rolling_average(self, df) -> DataFrame:
        """
        Fast rolling per-possession averages on each team's last X games.
        """
        df = df.copy()

        # sort once to guarantee correct rolling
        df = df.sort_values(["team_TeamID", "Season", "DayNum"]).reset_index(drop=True)

        roll = (
            df.groupby("team_TeamID")
            [self.feature_cols]
            .shift(1)
            .groupby(df["team_TeamID"])
            .rolling(window=self.rolling_window, min_periods=30)
            .mean()
            .reset_index(level=0, drop=True)
        )
        
        roll = roll.add_suffix(f"_avg")
        
        df = pd.concat([df[self.other_cols], roll], axis=1)

        return df.reset_index(drop=True)
    
    
    def apply_shrinkage(self, df_: DataFrame) -> DataFrame:
        """
        Applies regression-to-mean shrinkage:
            shrunk = w * rolling + (1 - w) * (league_mean_prev_season + team_mean_prev_season)/2
        """
        # All team-season averages (Season, team) already computed
        team = self.season_team_averages.copy()
        team = team.rename(columns={col: f"{col}_league" for col in self.feature_cols})
        # --- 1. Merge league averages ONCE (for previous season) ---
        df = df_.copy()
        df["Season_prev"] = df["Season"] - 1

        df = df.merge(
            team.rename(columns={"Season": "Season_prev"}),
            how="left",
            on=["Season_prev", "team_TeamID"]
#             suffixes=("", "_league")
        )
        
        # Fill only the league columns with 0
        league_cols = [c for c in df.columns if c.endswith("_league")]
        for col in self.feature_cols:
            # inaugural season does not have previous seaosn weighting applied
            df[col+"_league"] = df[col+"_league"].fillna(df[col+"_avg"])

        # Default: If no weight column exists, use scalar weight
        w = df["weight"] if "weight" in df.columns else 1

        # --- 3. Vectorized shrinkage for avg_cols ---
        for col in self.feature_cols:
            df[col+"_avg"] = w * df[col+"_avg"] + (1 - w) * df[col+"_league"]

        return df.drop(columns=league_cols+["Season_prev"])

    def fit_normalization(self, df: DataFrame) -> None:
        """
        Fit min/max or z-score normalization parameters.
        Best practice: save parameters that allow *future seasons* to be normalized identically.
        """
        #cols = [c for c in df.columns if "_per_poss" in c or "_shrunk" in c]

#         self.norm_params = {
#             c: {
#                 "mean": df[c].mean(),
#                 "std": df[c].std(),
#                 "min": df[c].min(),
#                 "max": df[c].max()
#             }
#             for c in self.feature_cols
#         }
        
        self.norm_params = {}
        for c in self.feature_cols:
            for suf in self.feature_methods:
                col = f"{c}_{suf}"
                self.norm_params[col] = {
                    "mean": df[col].mean(),
                    "std": df[col].std(),
                    "min": df[col].min(),
                    "max": df[col].max()
                }

    def transform_normalization(self, df: DataFrame, method: str = "z") -> DataFrame:
        """
        Normalize using fitted params.  
        method: "z" → z-score normalization  
                "minmax" → min/max scaling
        """
        if self.norm_params is None:
            raise ValueError("Call fit_normalization(df) first!")

        df = df.copy()

        for c, params in self.norm_params.items():
            if method == "z":
                df[c] = (df[c] - params["mean"]) / params["std"]
            else:
                df[c] = (df[c] - params["min"]) / (params["max"] - params["min"])

        return df

    
    def build_feature_table(self) -> DataFrame:
        """
        One-shot pipeline:
            1. per-possession stats
            2. rolling team stats
            3. shrinkage for early season
            4. normalization
        """
        df = self.per_possession.copy()
        df = self.make_recency_weight(df)
        df = self.get_rolling_average(df)
        df = self.apply_shrinkage(df)

        # Fit normalization on training-season only (avoid data leakage)
#         self.fit_normalization(df)
#         df = self.transform_normalization(df)

        return df

In [130]:
fb = FeatureBuilder(2025, True, rolling_window=30, feature_methods=['avg'])
processed_data = fb.per_possession

In [131]:
x = fb.build_feature_table()

In [132]:
x[x['team_TeamID']==1393][["Season", "DayNum", "team_Score_avg", "opponent_Score_avg"]].head(50)

,Season,DayNum,team_Score_avg,opponent_Score_avg
190582,2003,10,NaN,NaN
190583,2003,20,NaN,NaN
190584,2003,29,NaN,NaN
190585,2003,32,NaN,NaN
190586,2003,36,NaN,NaN
190587,2003,40,NaN,NaN
190588,2003,47,NaN,NaN
190589,2003,54,NaN,NaN
190590,2003,56,NaN,NaN
190591,2003,65,NaN,NaN


In [133]:
[1,2,3][-3]

1

In [48]:
list(x.columns)

['Season',
 'DayNum',
 'team_TeamID',
 'team_Score',
 'opponent_TeamID',
 'opponent_Score',
 'Loc',
 'NumOT',
 'team_FGM',
 'team_FGA',
 'team_FGM3',
 'team_FGA3',
 'team_FTM',
 'team_FTA',
 'team_OR',
 'team_DR',
 'team_Ast',
 'team_TO',
 'team_Stl',
 'team_Blk',
 'team_PF',
 'opponent_FGM',
 'opponent_FGA',
 'opponent_FGM3',
 'opponent_FGA3',
 'opponent_FTM',
 'opponent_FTA',
 'opponent_OR',
 'opponent_DR',
 'opponent_Ast',
 'opponent_TO',
 'opponent_Stl',
 'opponent_Blk',
 'opponent_PF',
 'Tournament',
 'GameID',
 'Possessions',
 'Minutes',
 'team_Score_per_poss',
 'opponent_Score_per_poss',
 'team_FGM_per_poss',
 'opponent_FGM_per_poss',
 'team_FGA_per_poss',
 'opponent_FGA_per_poss',
 'team_FGM3_per_poss',
 'opponent_FGM3_per_poss',
 'team_FGA3_per_poss',
 'opponent_FGA3_per_poss',
 'team_FTM_per_poss',
 'opponent_FTM_per_poss',
 'team_FTA_per_poss',
 'opponent_FTA_per_poss',
 'team_OR_per_poss',
 'opponent_OR_per_poss',
 'team_DR_per_poss',
 'opponent_DR_per_poss',
 'team_Ast_per

In [73]:

x[x['team_TeamID']==1393].sort_values(['Season', 'DayNum'])[['Season', 'DayNum', 'team_Score_per_poss_roll30_shrunk', 'team_Score_per_poss_roll30']]

,Season,DayNum,team_Score_per_poss_roll30_shrunk,team_Score_per_poss_roll30
3,2003,10,NaN,NaN
396,2003,20,1.060105,1.043050
1114,2003,29,1.209691,1.062608
1294,2003,32,1.132042,1.041946
1626,2003,36,1.248073,1.089731
...,...,...,...,...
239451,2025,117,1.059087,1.059087
239639,2025,120,0.968949,0.968949
240044,2025,124,0.965123,0.965123
240212,2025,127,1.044690,1.044690


In [61]:
x[x['team_TeamID']==1393].sort_values(['Season', 'DayNum']).reset_index(drop=True)[['team_Score_per_poss_roll30_shrunk']]

,team_Score_per_poss_roll30_shrunk
0,0.954995
1,1.082008
2,1.208467
3,1.141225
4,1.244059
...,...
772,1.074464
773,1.040034
774,1.069389
775,1.067338


In [ ]:
class FeatureBuilder:
    """
    `FeatureBuilder` for NCAA M/W Basketball data
    """
    #constants
    stat_cols = [
        'Score',
        'FGM',
        'FGA',
        'FGM3',
        'FGA3',
        'FTM',
        'FTA',
        'OR',
        'DR',
        'Ast',
        'TO',
        'Stl',
        'Blk',
        'PF',
#         'Poss'
    ]

    def __init__(self, year: int=2025, men: bool=True, rolling_window: int=30) -> None:
        """
        Initializes `FeatureBuilder` for a certain data pull year and gender

        Args:
            year: Year of the data pull
            men: boolean on if you want Mens (`True`) or Women's (`False`)

        Returns:
            `None`, but initilaizes dataloader object
        """
        self.year = year
        self.gender = "M" if men else "W"
        self.PROJECT_ROOT = Path().resolve().parent  # /src/data -> project root
        self.feature_cols = self.get_feature_cols("all")
        self.rolling_window = rolling_window
        
    @lazy
    def data(self) -> DataFrame:
        path = (
                self.PROJECT_ROOT
                / "data"
                / "interim"
                / f"{self.gender}{self.year}_formatted_data.csv"
            )
        if not os.path.exists(path):
            raise ValueError(f"{path} does not exist. Please run the DataLoader and write the processed data first")
        return pd.read_csv(path).sort_values(["Season", "DayNum", "GameID"]).reset_index(drop=True)
    
    @lazy
    def per_possession(self) -> DataFrame:
        df = self.data.copy()
        for col in self.stat_cols:
            if col == "Poss":
                continue
            for team in ['team', 'opponent']:
                df[f'{team}_{col}_per_poss'] = df[f'{team}_{col}']/df.Possessions
        df[f'Possessions_per_min'] = df.Possessions/df.Minutes
        return df
    
#     @lazy #lazy loader for caching 
#     def season_averages(self) -> DataFrame:
#         df = self.data.copy()
#         return df.groupby("Season").average...
    
    # Save min/max, std, etc? for normalization? Best practice here?
    # Save season by season stats for rolling average warm up for new, next season
    # Rolling average of stats over x amount of games or days
        # when new season: μ^=w⋅xˉ+(1−w)⋅μleague - to account for low sample size
        # or regress to the mean of last year (+league average)
        # standard deviation of these rolling average for simulations?
    # 
    
    def get_feature_cols(self, include: str = "all") -> list:
        """
        Flexible selector for all engineered feature columns.

        include:
            "raw"         - raw stats like FGM, PF, TO
            "per_poss"    - per-possession versions of raw stats
            "all"         - everything
        """

        cols = []

        # -----------------------------
        # RAW FEATURES (from stat_cols)
        # -----------------------------
        if include in ("raw", "all"):
            # stat_cols = ['FGM','FGA','PF',...]
            for stat in self.stat_cols:
                cols.append(f"team_{stat}")
                cols.append(f"opponent_{stat}")

        # -----------------------------------------
        # PER-POSSESSION FEATURES (added by pipeline)
        # -----------------------------------------
        if include in ("per_poss", "all"):
            for stat in self.stat_cols:
                cols.append(f"team_{stat}_per_poss")
                cols.append(f"opponent_{stat}_per_poss")
            cols.append("Possessions_per_min")
            

        return cols

    
    # -------------------------------
    # ========== SEASON AVERAGES ==========
    # -------------------------------
    @lazy
    def season_averages(self) -> DataFrame:
        """
        League-wide seasonal averages for per-possession stats.
        Used for shrinkage on low sample sizes.
        """
        df = self.per_possession.copy()

        
        league = (
            df.groupby("Season")[self.feature_cols]
            .mean()
            .reset_index()
        )
        return league
    
    @lazy
    def season_team_averages(self) -> DataFrame:
        """
        League-wide seasonal averages for per-possession stats.
        Used for shrinkage on low sample sizes.
        """
        df = self.per_possession.copy()

        # 1. Make league per-team per-season averages
        league = (
            df.groupby(["Season", "team_TeamID"])[self.feature_cols]
            .mean()
            .reset_index()
        )#.set_index("Season")

        # 2. Make season-wide averages
        season_averages = self.season_averages.copy().set_index("Season")

        # 3. Merge league with season averages on Season
        league = league.merge(
            season_averages[self.feature_cols],
            left_on="Season",
            right_index=True,
            suffixes=("", "_season")
        )

        # 4. Compute simple average (regression to mean)
        for col in self.feature_cols:
            league[col] = (league[col] + league[f"{col}_season"]) / 2

        # 5. Optionally drop the "_season" columns
        league.drop(columns=[f"{c}_season" for c in self.feature_cols], inplace=True)
        return league

    
    
    def make_recency_weight(self, df):
        df = df.copy()
        
        min_day = (df.groupby("Season")["DayNum"]
            .min()
            .reset_index()
            .set_index('Season')
        ) 
        
        
        lin_func = np.linspace(1, 5, 17)/5
        recency_func = lambda x: lin_func.max() if x >= len(lin_func) else lin_func[x]
        df["weight"] = (((df.DayNum - df.Season.map(min_day['DayNum']))//7)
                       ).apply(recency_func)
        return df

    
    # -------------------------------
    # ========== TEAM ROLLING STATS ==========
    # -------------------------------
    def make_team_rolling(self, df, window: int = 30, min_periods: int = 20) -> DataFrame:
        """
        Fast rolling per-possession averages on each team's last X games.
        """
        df = df.copy()

        # sort once to guarantee correct rolling
        df = df.sort_values(["team_TeamID", "Season", "DayNum"]).reset_index(drop=True)

        # shift once BEFORE rolling
#         shifted = df.groupby("team_TeamID")[self.feature_cols].shift(1)

        # rolling using groupby.rolling — fast, vectorized C implementation
        roll = (
            df.groupby("team_TeamID")
            [self.feature_cols]
            .shift(1)
            .groupby(df["team_TeamID"])
            .rolling(window=window, min_periods=min_periods)
            .mean()
            .reset_index(level=0, drop=True)
        )

        # rename columns
        roll = roll.add_suffix(f"_roll{window}")

        # join once
        df = pd.concat([df, roll], axis=1)

        return df.reset_index(drop=True)
    


    # -------------------------------
    # ========== SHRINKAGE METHOD ==========
    # -------------------------------
    def apply_shrinkage(self, df: DataFrame, window: int = 5, weight: float = 0.5) -> DataFrame:
        """
        Applies regression-to-mean shrinkage:
            shrunk = w * rolling + (1 - w) * league_mean_prev_season
        """
        # All team-season averages (Season, team) already computed
        team = self.season_team_averages.copy()

        # --- 1. Merge league averages ONCE (for previous season) ---
        df = df.copy()
        df["Season_prev"] = df["Season"] - 1

        df = df.merge(
            team.rename(columns={"Season": "Season_prev"}),
            how="left",
            on=["Season_prev", "team_TeamID"],
            suffixes=("", "_league")
        )
        # Fill only the league columns with 0
        league_cols = [c for c in df.columns if c.endswith("_league")]
        df[league_cols] = df[league_cols].fillna(0)

        # --- 2. Identify rolling and base columns ---
        roll_cols = [c for c in df.columns if f"roll{window}" in c]
        base_cols = [c.replace(f"_roll{window}", "") for c in roll_cols]

        # Default: If no weight column exists, use scalar weight
        w = df["weight"] if "weight" in df.columns else weight

        # --- 3. Vectorized shrinkage for all columns ---
        for roll_col, base_col in zip(roll_cols, base_cols):
            league_col = base_col  # after merge, league average lives here
            df[f"{roll_col}_shrunk"] = w * df[roll_col] + (1 - w) * df[league_col]

        
        
        return df.drop(columns=league_cols)

    # -------------------------------
    # ========== NORMALIZATION ==========
    # -------------------------------
    def fit_normalization(self, df: DataFrame) -> None:
        """
        Fit min/max or z-score normalization parameters.
        Best practice: save parameters that allow *future seasons* to be normalized identically.
        """
        cols = [c for c in df.columns if "_per_poss" in c or "_shrunk" in c]

        self.norm_params = {
            c: {
                "mean": df[c].mean(),
                "std": df[c].std(),
                "min": df[c].min(),
                "max": df[c].max()
            }
            for c in cols
        }

    def transform_normalization(self, df: DataFrame, method: str = "z") -> DataFrame:
        """
        Normalize using fitted params.  
        method: "z" → z-score normalization  
                "minmax" → min/max scaling
        """
        if self.norm_params is None:
            raise ValueError("Call fit_normalization(df) first!")

        df = df.copy()

        for c, params in self.norm_params.items():
            if method == "z":
                df[c + "_norm"] = (df[c] - params["mean"]) / params["std"]
            else:
                df[c + "_norm"] = (df[c] - params["min"]) / (params["max"] - params["min"])

        return df

    # -------------------------------
    # ========== FULL PIPELINE ==========
    # -------------------------------
    def build_feature_table(self) -> DataFrame:
        """
        One-shot pipeline:
            1. per-possession stats
            2. rolling team stats
            3. shrinkage for early season
            4. normalization
        """
        df = self.per_possession.copy()
        df = self.make_recency_weight(df)
        df = self.make_team_rolling(df, window=self.rolling_window)
        df = self.apply_shrinkage(df, window=self.rolling_window, weight=0.6)

        # Fit normalization on training-season only (avoid data leakage)
#         self.fit_normalization(df)
#         df = self.transform_normalization(df)

        return df